<a href="https://colab.research.google.com/github/vika3mm/gost_progect/blob/main/%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_layout3_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install --no-build-isolation seqeval

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import random
import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer
)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_dataset_2.json"
IMAGES_DIR = "/content/drive/MyDrive/GOST_Project/общие_фото_новинка"
MODEL_OUTPUT = "/content/drive/MyDrive/GOST_Project/layoutlm_model_split"

# Путь для нового датасета (разбитого на части)
SPLIT_DATASET_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_dataset_split.json"

In [ ]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Страниц в исходном датасете:", len(data))

labels = sorted(
    list(
        set(
            word["label"]
            for page in data
            for word in page["words"]
        )
    )
)
print("Метки:", labels)
print("Количество меток:", len(labels))

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

Страниц в исходном датасете: 327
Метки: ['BODY', 'FIGURE', 'FIGURE_CAPTION', 'FOOTNOTE', 'FORMULA', 'OTHER', 'REFERENCE', 'SECTION_HEADER', 'TABLE', 'TABLE_CAPTION', 'TITLE', 'TOC']
Количество меток: 12
{'BODY': 0, 'FIGURE': 1, 'FIGURE_CAPTION': 2, 'FOOTNOTE': 3, 'FORMULA': 4, 'OTHER': 5, 'REFERENCE': 6, 'SECTION_HEADER': 7, 'TABLE': 8, 'TABLE_CAPTION': 9, 'TITLE': 10, 'TOC': 11}


In [ ]:
MAX_WORDS_PER_CHUNK = 400

def split_long_pages(data, max_words=MAX_WORDS_PER_CHUNK):
    """
    Разбивает длинные страницы на части.
    Каждая часть становится отдельным примером.
    """
    split_data = []
    total_original = len(data)
    total_split = 0

    for page in data:
        words = page["words"]
        image_name = page["image"]

        if len(words) <= max_words:
            # Страница короткая — оставляем как есть
            split_data.append({
                "image": image_name,
                "words": words
            })
            total_split += 1
        else:
            # Страница длинная — режем на части
            num_chunks = (len(words) + max_words - 1) // max_words
            for i in range(num_chunks):
                start = i * max_words
                end = min((i + 1) * max_words, len(words))
                chunk_words = words[start:end]

                split_data.append({
                    "image": image_name,
                    "words": chunk_words,
                    "chunk_index": i,
                    "chunk_total": num_chunks
                })
                total_split += 1

    print(f"Исходных страниц: {total_original}")
    print(f"После разбиения: {total_split}")
    print(f"Добавлено частей: {total_split - total_original}")
    return split_data

split_data = split_long_pages(data, max_words=MAX_WORDS_PER_CHUNK)

Исходных страниц: 327
После разбиения: 327
Добавлено частей: 0


In [ ]:
with open(SPLIT_DATASET_PATH, "w", encoding="utf-8") as f:
    json.dump(split_data, f, ensure_ascii=False)

print(f"Сохранено: {SPLIT_DATASET_PATH}")
print(f"Всего примеров: {len(split_data)}")

Сохранено: /content/drive/MyDrive/GOST_Project/layoutlm_dataset_split.json
Всего примеров: 327


In [ ]:
import re

def get_document_name(image_name):
    """
    Извлекает имя документа из имени файла.
    Пример: 'alekseeva_e_a_page_1.png' -> 'alekseeva_e_a'
    """
    # Убираем расширение
    name = os.path.splitext(image_name)[0]
    # Убираем '_page_N' в конце
    name = re.sub(r'_page_\d+$', '', name)
    return name

# Собираем уникальные документы
documents = {}
for item in split_data:
    doc_name = get_document_name(item["image"])
    if doc_name not in documents:
        documents[doc_name] = []
    documents[doc_name].append(item)

doc_names = sorted(documents.keys())
print(f"Уникальных документов: {len(doc_names)}")
print(doc_names[:10])

train_docs, test_docs = train_test_split(
    doc_names, test_size=0.15, random_state=42
)
train_docs, val_docs = train_test_split(
    train_docs, test_size=0.15, random_state=42
)

print(f"Train документов: {len(train_docs)}")
print(f"Val документов: {len(val_docs)}")
print(f"Test документов: {len(test_docs)}")

train_data = [item for doc in train_docs for item in documents[doc]]
val_data = [item for doc in val_docs for item in documents[doc]]
test_data = [item for doc in test_docs for item in documents[doc]]

print(f"Train примеров: {len(train_data)}")
print(f"Val примеров: {len(val_data)}")
print(f"Test примеров: {len(test_data)}")


Уникальных документов: 19
['alekseeva_e_a', 'tohtueva_e_a', 'trubacheva_v_p', 'Анисимова_текст', 'Багрянова_текст', 'Вихорев_Текст', 'Воробьев_ТекстВКР', 'Габитов_диплом', 'Гащенко_диплом', 'Головин_диплом']
Train документов: 13
Val документов: 3
Test документов: 3
Train примеров: 175
Val примеров: 86
Test примеров: 66


In [ ]:
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

In [ ]:
def normalize_bbox(bbox, width, height):
    return [
        int(1000 * bbox[0] / width),
        int(1000 * bbox[1] / height),
        int(1000 * bbox[2] / width),
        int(1000 * bbox[3] / height),
    ]

def prepare_examples(pages):
    dataset = []
    for page in pages:
        image_path = os.path.join(IMAGES_DIR, page["image"])
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        words = []
        boxes = []
        labels = []

        for word in page["words"]:
            words.append(word["text"])
            boxes.append(normalize_bbox(word["bbox"], width, height))
            labels.append(label2id[word["label"]])

        dataset.append({
            "image": image,
            "words": words,
            "boxes": boxes,
            "labels": labels
        })
    return dataset

train_examples = prepare_examples(train_data)
val_examples = prepare_examples(val_data)
test_examples = prepare_examples(test_data)

print(len(train_examples))
print(len(val_examples))
print(len(test_examples))

175
86
66


In [ ]:
def encode_example(example):
    encoding = processor(
        images=example["image"],
        text=example["words"],
        boxes=example["boxes"],
        word_labels=example["labels"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    encoding = {k: v.squeeze() for k, v in encoding.items()}
    return encoding

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)
test_ds = Dataset.from_list(test_examples)

print(train_ds)

train_ds = train_ds.map(encode_example, remove_columns=train_ds.column_names)
val_ds = val_ds.map(encode_example, remove_columns=val_ds.column_names)
test_ds = test_ds.map(encode_example, remove_columns=test_ds.column_names)

print(train_ds)

Dataset({
    features: ['image', 'words', 'boxes', 'labels'],
    num_rows: 175
})


Map:   0%|          | 0/175 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask', 'bbox', 'pixel_values'],
    num_rows: 175
})


In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.bias      | MISSING | 
classifier.dense.weight    | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
!pip show seqeval

Name: seqeval
Version: 0.0.10
Summary: Testing framework for sequence labeling
Home-page: https://github.com/chakki-works/seqeval
Author: Hironsan
Author-email: hiroki.nakayama.py@gmail.com
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: Keras, numpy
Required-by: 


In [ ]:
!pip install --upgrade --no-build-isolation seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/9d/2d/233c79d5b4e5ab1dbf111242299153f3caddddbb691219f363ad55ce783d/seqeval-1.2.2.tar.gz (from https://pypi.org/simple/seqeval/): Requested seqeval from https://files.pythonhosted.org/packages/9d/2d/233c79d5b4e5ab1dbf111242299153f3caddddbb691219f363ad55ce783d/seqeval-1.2.2.tar.gz has inconsistent version: expected '1.2.2', but metadata has '0.0.0'
  Using cached seqeval-1.2.1.tar.gz (43 kB)
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/c4/47/f85f522f1f2532ca650474088e4024a9f52d524379bff058eaadf53cb663/seqeval-1.2.1.tar.gz (from https://pypi.org/simple/seqeval/): Requested seqeval from https://files.pythonhosted.org/packages/c4/47/f85f522f1f2532ca650474088e4024a9f52d524379bff058eaadf53cb663/seqeval-1.2.1.tar.gz has inconsistent version: expected '1.2.1', but metadata has '0.0.0'
  Using cached seq

In [ ]:
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        cur_preds = []
        cur_labels = []
        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            cur_preds.append(id2label[p_i])
            cur_labels.append(id2label[l_i])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": None,  # seqeval не даёт accuracy напрямую
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    save_total_limit=2,
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.184433,1.018403,0.788321,0.607424,0.686150,None
2,0.720063,0.472966,0.840131,0.868954,0.854299,None
3,0.263583,0.419441,0.861081,0.895951,0.878170,None
4,0.063846,0.430211,0.847665,0.898200,0.872201,None
5,0.045756,0.445201,0.880992,0.899325,0.890064,None
6,0.121238,0.313646,0.900598,0.932508,0.916275,None
7,0.037134,0.339149,0.915084,0.921260,0.918161,None
8,0.032289,0.279219,0.928971,0.934196,0.931576,None
9,0.005891,0.262881,0.929372,0.932508,0.930938,None
10,0.044043,0.283062,0.931818,0.945444,0.938582,None


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1320, training_loss=0.17539189979292905, metrics={'train_runtime': 2484.2548, 'train_samples_per_second': 1.057, 'train_steps_per_second': 0.531, 'total_flos': 696757565952000.0, 'train_loss': 0.17539189979292905, 'epoch': 15.0})

In [ ]:
trainer.save_model(MODEL_OUTPUT)
processor.save_pretrained(MODEL_OUTPUT)
print(f"Модель сохранена: {MODEL_OUTPUT}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена: /content/drive/MyDrive/GOST_Project/layoutlm_model_split


In [ ]:
metrics = trainer.evaluate(test_ds)
print(metrics)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.004016,0.638919,15,0.946058,0.862004,0.902077,None


{'eval_loss': 0.6389191150665283, 'eval_precision': 0.946058091286307, 'eval_recall': 0.8620037807183365, 'eval_f1': 0.9020771513353116, 'eval_accuracy': None}


In [ ]:
print(trainer.state.best_model_checkpoint)